# VÍA VIVA · 01 — Recorridos de phyphox: espectro, características y primer clasificador

Flujo: **CSV de phyphox → remuestreo → eje vertical → espectro (elección de fs) → filtro → ventanas por energía/jerk → características → clasificador (leave-one-drive-out)**.

* Las funciones están en `senales.py`; este notebook solo las llama y grafica.
* Datos reales: poner los recorridos en `data/raw/phyphox/` (formato en `analysis/README.md`). Esa carpeta **no** se sube a Git.
* **Si no hay datos reales, el notebook usa recorridos SINTÉTICOS** solo para comprobar que el código corre. Los números que salgan así **no son resultados** y no van al informe.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import senales as s

CARPETA = Path("../data/raw/phyphox")
hay_reales = CARPETA.exists() and any(not p.name.endswith("_etiquetas.csv") for p in CARPETA.iterdir())
print("Carpeta de datos:", CARPETA.resolve())
print("Datos reales encontrados:", hay_reales)

## 1. Cargar y procesar los recorridos

In [ ]:
if hay_reales:
    salida = s.procesar_carpeta(CARPETA)
    recorridos, ventanas = salida["recorridos"], salida["ventanas"]
    SINTETICO = False
else:
    print("ATENCIÓN: SIN DATOS REALES. Se generan 4 recorridos SINTÉTICOS. Nada de lo que sigue es un resultado.")
    recorridos = []
    for i in range(4):
        acc, gps, et = s.recorrido_sintetico(f"sintetico_{i}", semilla=i)
        recorridos.append(s.procesar_recorrido(acc, gps, f"sintetico_{i}", et))
    ventanas = pd.concat([r.ventanas for r in recorridos], ignore_index=True)
    SINTETICO = True

pd.DataFrame([{
    "recorrido": r.nombre,
    "duracion_s": round(r.t[-1] - r.t[0], 1),
    "fs_telefono_hz": round(r.fs_nativa, 1),
    "fs_trabajo_hz": r.fs,
    "ventanas": len(r.ventanas),
    "con_gps": r.gps is not None,
} for r in recorridos])

## 2. Señal vertical y eventos detectados

In [ ]:
r = recorridos[0]
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(r.t, r.a_vert, lw=0.5, color="0.3", label="a vertical filtrada")
det = r.ventanas[r.ventanas.origen == "detectada"]
for clase, color in [("hueco", "tab:red"), ("resalto", "tab:orange"), ("impacto_fuerte", "tab:purple"), ("normal", "tab:green")]:
    sel = det[det.clase == clase]
    ax.scatter(sel.t_s, np.zeros(len(sel)), color=color, s=25, zorder=3, label=f"{clase} ({len(sel)})")
sin = det[det.clase.isna()]
if len(sin):
    ax.scatter(sin.t_s, np.zeros(len(sin)), color="k", marker="x", s=25, zorder=3, label=f"sin etiqueta ({len(sin)})")
ax.set(xlabel="tiempo (s)", ylabel="m/s²", title=f"{r.nombre}{' (SINTÉTICO)' if SINTETICO else ''}")
ax.legend(loc="upper right", fontsize=8, ncol=6)
plt.tight_layout()

## 3. Espectro y elección de la frecuencia de muestreo

Se busca la frecuencia bajo la cual queda el **95 % de la energía** de la señal vertical (sin filtrar). Por Nyquist la fs mínima es el doble; se recomienda 2,5× para dejar espacio al filtro antialias. Hipótesis del diseño: 200 Hz con antialias cerca de 80 Hz.

Ojo: si el teléfono muestrea a menos de 200 Hz, el espectro solo llega a fs/2 y la conclusión vale hasta ahí.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.5))
for r in recorridos:
    a1.semilogy(r.espectro.f, r.espectro.psd, lw=0.8, label=r.nombre)
    a2.plot(r.espectro.f, np.cumsum(r.espectro.psd) / r.espectro.psd.sum(), lw=0.8)
a2.axhline(0.95, color="k", ls="--", lw=0.8)
a1.set(xlabel="Hz", ylabel="PSD (m/s²)²/Hz", title="Densidad espectral (Welch)")
a2.set(xlabel="Hz", ylabel="energía acumulada", title="Energía acumulada")
a1.legend(fontsize=7)
plt.tight_layout()

tabla_fs = pd.DataFrame([{
    "recorrido": r.nombre,
    "f_95%_hz": round(r.espectro.f_energia, 1),
    "fs_minima_hz": round(r.espectro.fs_minima, 1),
    "fs_recomendada_hz": round(r.espectro.fs_recomendada, 1),
} for r in recorridos])
print("fs recomendada (máximo entre recorridos):", tabla_fs["fs_recomendada_hz"].max(), "Hz",
      "(SINTÉTICO: no concluir nada)" if SINTETICO else "")
tabla_fs

## 4. Filtro: antes y después en un evento

In [ ]:
r = recorridos[0]
ev = r.ventanas[r.ventanas.origen == "detectada"].iloc[0]
i = int(np.searchsorted(r.t, ev.t_s))
w = slice(max(0, i - int(r.fs)), i + int(r.fs))
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(r.t[w], r.a_sin_filtro[w], lw=0.6, color="0.6", label="sin filtrar")
ax.plot(r.t[w], r.a_vert[w], lw=1.2, label=f"pasa-bajos Butterworth {min(80, 0.4 * r.fs):g} Hz, fase cero")
ax.axvline(ev.t_s, color="r", ls=":", label=f"evento: {ev.clase}")
ax.set(xlabel="tiempo (s)", ylabel="m/s²")
ax.legend(fontsize=8)
plt.tight_layout()

## 5. Características por clase

Pico, RMS, curtosis, factor de cresta, jerk, asimetría, energía por banda y las de amplitud divididas por la velocidad del GPS.

In [ ]:
etiquetadas = ventanas.dropna(subset=["clase"])
print(etiquetadas["clase"].value_counts().to_string())
etiquetadas.groupby("clase")[["pico", "rms", "curtosis", "factor_cresta", "jerk_max", "pico_por_vel"]].median().round(3)

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(14, 3.2))
orden = [c for c in s.CLASES if c in set(etiquetadas.clase)]
for ax, col in zip(axs, ["pico", "curtosis", "factor_cresta", "energia_15_30hz"]):
    ax.boxplot([etiquetadas.loc[etiquetadas.clase == c, col].dropna() for c in orden], tick_labels=orden)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=45, labelsize=8)
plt.tight_layout()

## 6. Primer clasificador (leave-one-drive-out)

Se entrena con todos los recorridos menos uno y se prueba en el que quedó fuera, rotando. **Nunca** se separa por ventana: las ventanas de un mismo recorrido se parecen demasiado e inflarían el F1.

* `bosque`: bosque aleatorio (referencia).
* `arbol`: árbol de profundidad 4, fácil de exportar a C para el nodo.

In [ ]:
con_etiqueta = ventanas.dropna(subset=["clase"])
if con_etiqueta["recorrido"].nunique() < 2:
    print("Se necesitan al menos 2 recorridos con etiquetas para leave-one-drive-out.")
    res = None
else:
    res = s.entrenar_lodo(con_etiqueta)
    print(f"{res['n_ventanas']} ventanas de {res['n_recorridos']} recorridos", "(SINTÉTICO)" if SINTETICO else "")
    for nombre in ("bosque", "arbol"):
        print(f"\n=== {nombre}: F1 macro = {res[nombre]['f1_macro']:.3f} · F1 hueco = {res[nombre]['f1_hueco']:.3f}")
        print(res[nombre]["reporte"])

In [ ]:
if res is not None:
    fig, axs = plt.subplots(1, 2, figsize=(11, 4))
    cm = res["bosque"]["confusion"]
    axs[0].imshow(cm, cmap="Blues")
    axs[0].set(xticks=range(len(cm)), yticks=range(len(cm)), xticklabels=cm.columns, yticklabels=cm.index,
               xlabel="predicha", ylabel="real", title="Matriz de confusión (bosque, LODO)")
    axs[0].tick_params(axis="x", rotation=45)
    for (i, j), v in np.ndenumerate(cm.to_numpy()):
        axs[0].text(j, i, v, ha="center", va="center")
    imp = pd.Series(res["bosque"]["modelo_final"][-1].feature_importances_, index=res["columnas"]).sort_values()
    imp.tail(10).plot.barh(ax=axs[1], title="Importancia de características (bosque)")
    plt.tight_layout()

## 7. Siguientes pasos

1. Grabar los primeros recorridos con phyphox a la tasa más alta del teléfono y anotar los eventos (`<recorrido>_etiquetas.csv`).
2. Correr este notebook con datos reales y decidir fs con la tabla de la sección 3.
3. Repetir con los CSV del nodo (`crudo_NNN.csv`, `etiquetas_NNN.csv`) cuando llegue el hardware.
4. Exportar el árbol a C (o entrenar la CNN 1D int8) y medir E3 en el ESP32-S3.